In [1]:
# 01_eda_preprocess.ipynb
# Give Me Some Credit (GMSC) — EDA, cleaning, and monotonic WoE binning.
# Produces the shared WoE assets consumed by 02 (logit-WoE vs stump isomorphism)
# and all downstream notebooks. Single-cell convention per project rules.
#
# WoE SIGN CONVENTION (credit standard): WoE(bin) = ln( bad%_bin / good%_bin ),
# so HIGHER risk -> HIGHER WoE and logistic beta on WoE is POSITIVE. This matches
# the walkthrough doc where score decreases as WoE rises via a negative Factor.
#
# Figures: seaborn greyscale, dpi 600, png + pdf, no captions (project rule).

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='paper')
sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor': '0.2', 'grid.color': '0.85', 'font.size': 10})

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data')
TABLES = os.path.join(ROOT, 'results', 'tables')
FIGS   = os.path.join(ROOT, 'results', 'figures')
for d in (DATA, TABLES, FIGS):
    os.makedirs(d, exist_ok=True)

def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(FIGS, name + '.' + ext), dpi=600, bbox_inches='tight')
    plt.close(fig)

TARGET = 'SeriousDlqin2yrs'
MIN_BIN_FRAC = 0.01     # bins (incl. missing) with < 1% of rows are neutralized / merged
IV_THRESHOLD = 0.02    # features with IV < this are dropped (credit-standard weak-predictor cut)

# ===================================================================
# 1. LOAD
# ===================================================================
df = pd.read_csv(os.path.join(DATA, 'cs-training.csv'), index_col=0).reset_index(drop=True)
features = [c for c in df.columns if c != TARGET]
print('Loaded GMSC:', df.shape, '| features:', len(features))
print('Target rate: %.4f (bad=%d / total=%d)' % (
    df[TARGET].mean(), int(df[TARGET].sum()), len(df)))

# ===================================================================
# 2. CLEAN  (sentinels -> NaN, impossible values -> NaN)
# ===================================================================
dfc = df.copy()
pastdue = ['NumberOfTime30-59DaysPastDueNotWorse',
           'NumberOfTime60-89DaysPastDueNotWorse',
           'NumberOfTimes90DaysLate']
sentinel_counts = {}
for c in pastdue:
    mask = dfc[c] >= 96
    sentinel_counts[c] = int(mask.sum())
    dfc.loc[mask, c] = np.nan
age0 = int((dfc['age'] == 0).sum())
dfc.loc[dfc['age'] == 0, 'age'] = np.nan
print('\nSentinel(>=96) ->NaN:', sentinel_counts, '| age==0 ->NaN:', age0)

# ===================================================================
# 3. MONOTONIC WoE BINNING  (credit sign convention)
#    WoE(bin) = ln( (bad_b/Bad) / (good_b/Good) ),  Laplace-smoothed.
#    Missing is a separate bin ONLY if it holds >= MIN_BIN_FRAC of rows;
#    otherwise it is neutralized (WoE=0) so a 1-row bin cannot distort s(x).
# ===================================================================
TOTAL_GOOD = int((dfc[TARGET] == 0).sum())
TOTAL_BAD  = int((dfc[TARGET] == 1).sum())
MIN_N = int(MIN_BIN_FRAC * len(dfc))

def tree_edges(x, y, max_leaf=6, min_frac=0.05):
    m = x.notna().values
    xx = x[m].values.reshape(-1, 1); yy = y[m].values
    if len(np.unique(xx)) <= 1:
        return []
    t = DecisionTreeClassifier(max_leaf_nodes=max_leaf,
                               min_samples_leaf=int(min_frac*len(xx)),
                               random_state=SEED)
    t.fit(xx, yy)
    thr = t.tree_.threshold[t.tree_.feature >= 0]
    return sorted(float(v) for v in thr)

def bin_table(x, y, edges):
    cuts = [-np.inf] + list(edges) + [np.inf]
    idx = pd.cut(x, bins=cuts, labels=False, include_lowest=True)
    rows = []
    for b in range(len(cuts)-1):
        sel = (idx == b)
        rows.append([b, cuts[b], cuts[b+1],
                     int(((y==0)&sel).sum()), int(((y==1)&sel).sum())])
    selm = x.isna()
    rows.append([-1, np.nan, np.nan,
                 int(((y==0)&selm).sum()), int(((y==1)&selm).sum())])
    tb = pd.DataFrame(rows, columns=['bin','lo','hi','good','bad'])
    tb = tb[(tb['good']+tb['bad']) > 0].reset_index(drop=True)
    gr = (tb['good'] + 0.5) / (TOTAL_GOOD + 0.5)
    br = (tb['bad']  + 0.5) / (TOTAL_BAD  + 0.5)
    tb['woe'] = np.log(br / gr)                        # <-- credit convention
    tb['n'] = tb['good'] + tb['bad']
    tb['bad_rate'] = tb['bad'] / tb['n']
    tb['iv_part'] = (br - gr) * tb['woe']
    # neutralize sparse missing bin
    sparse = (tb['bin'] == -1) & (tb['n'] < MIN_N)
    tb.loc[sparse, 'woe'] = 0.0
    tb.loc[sparse, 'iv_part'] = 0.0
    return tb

def enforce_monotone(x, y, edges):
    e = list(edges)
    while len(e) > 0:
        tb = bin_table(x, y, e)
        fin = tb[tb['bin'] >= 0].sort_values('lo')
        br = fin['bad_rate'].values
        if len(br) <= 2 or np.all(np.diff(br) >= -1e-12) or np.all(np.diff(br) <= 1e-12):
            return e
        d = np.diff(br); s = np.sign(np.nansum(d)); viol = -s * d
        e.pop(int(np.argmax(viol)))
    return e

woe_maps, iv_summary, woe_long = {}, [], []
for c in features:
    edges = enforce_monotone(dfc[c], dfc[TARGET], tree_edges(dfc[c], dfc[TARGET]))
    tb = bin_table(dfc[c], dfc[TARGET], edges)
    tb.insert(0, 'feature', c)
    woe_maps[c] = tb.to_dict('records')
    has_miss = int(((tb['bin']==-1) & (tb['woe']!=0)).any())
    iv_summary.append([c, float(tb['iv_part'].sum()), int((tb['bin']>=0).sum()), has_miss])
    woe_long.append(tb)

iv_df = pd.DataFrame(iv_summary, columns=['feature','IV','n_bins','has_missing_bin']
                     ).sort_values('IV', ascending=False).reset_index(drop=True)
woe_long = pd.concat(woe_long, ignore_index=True)
print('\n=== Information Value (sorted) ===')
print(iv_df.to_string(index=False))

# --- feature selection by IV (drop weak predictors) ---
iv_df['selected'] = iv_df['IV'] >= IV_THRESHOLD
selected_features = iv_df.loc[iv_df['selected'], 'feature'].tolist()
dropped_features  = iv_df.loc[~iv_df['selected'], 'feature'].tolist()
print('\nIV threshold = %.3f | kept %d / %d features' % (IV_THRESHOLD, len(selected_features), len(features)))
print('Dropped (IV < thr):', dropped_features)
features = selected_features                      # downstream uses selected only
woe_maps = {c: woe_maps[c] for c in features}
woe_long = woe_long[woe_long['feature'].isin(features)].reset_index(drop=True)

# ===================================================================
# 4. APPLY WoE
# ===================================================================
def apply_woe(x, recs):
    out = pd.Series(np.nan, index=x.index)
    miss_woe = next((r['woe'] for r in recs if r['bin'] == -1), 0.0)
    out[x.isna()] = miss_woe
    for r in recs:
        if r['bin'] == -1:
            continue
        sel = (~x.isna()) & (x > r['lo']) & (x <= r['hi'])
        out[sel] = r['woe']
    return out.fillna(0.0)

woe_X = pd.DataFrame({c: apply_woe(dfc[c], woe_maps[c]) for c in features})
woe_X[TARGET] = dfc[TARGET].values
assert woe_X.drop(columns=[TARGET]).isna().sum().sum() == 0
print('\nWoE design matrix:', woe_X.shape, '| any NaN:', bool(woe_X.isna().any().any()))
print('Sanity: WoE should correlate POSITIVELY with bin bad_rate ->',
      'corr(woe,bad_rate)=%.3f' % woe_long.loc[woe_long.bin>=0, ['woe','bad_rate']].corr().iloc[0,1])

# ===================================================================
# 5. SAVE SHARED ASSETS
# ===================================================================
woe_X.to_parquet(os.path.join(DATA, 'gmsc_woe.parquet'))
dfc.to_parquet(os.path.join(DATA, 'gmsc_clean.parquet'))
with open(os.path.join(DATA, 'woe_maps.json'), 'w') as f:
    json.dump({'target': TARGET, 'features': features,
               'total_good': TOTAL_GOOD, 'total_bad': TOTAL_BAD,
               'woe_convention': 'ln(bad_pct/good_pct); higher risk -> higher WoE',
               'iv_threshold': IV_THRESHOLD,
               'dropped_features': dropped_features,
               'maps': woe_maps}, f, indent=2, default=float)
iv_df.to_csv(os.path.join(TABLES, '01_information_value.csv'), index=False)
woe_long.to_csv(os.path.join(TABLES, '01_woe_bins.csv'), index=False)
print('Saved: gmsc_woe.parquet, gmsc_clean.parquet, woe_maps.json,',
      '01_information_value.csv, 01_woe_bins.csv')

# ===================================================================
# 6. FIGURE — WoE curves (top-6 by IV), greyscale dpi600 png+pdf
# ===================================================================
top6 = iv_df['feature'].head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.2))
for ax, c in zip(axes.ravel(), top6):
    tb = woe_long[(woe_long['feature']==c) & (woe_long['bin']>=0)].sort_values('lo')
    ax.bar(range(len(tb)), tb['woe'], color='0.45', edgecolor='0.0', linewidth=.5)
    ax.axhline(0, color='0.1', lw=.6)
    ax.set_title('%s  (IV=%.3f)' % (c, iv_df.loc[iv_df.feature==c,'IV'].values[0]), fontsize=9)
    ax.set_xlabel('bin (low \u2192 high)'); ax.set_ylabel('WoE')
    ax.set_xticks(range(len(tb)))
fig.tight_layout()
savefig(fig, '01_woe_curves')
print('Saved figure: 01_woe_curves (png+pdf)')
print('\n01 complete.')


Loaded GMSC: (150000, 11) | features: 10
Target rate: 0.0668 (bad=10026 / total=150000)

Sentinel(>=96) ->NaN: {'NumberOfTime30-59DaysPastDueNotWorse': 269, 'NumberOfTime60-89DaysPastDueNotWorse': 269, 'NumberOfTimes90DaysLate': 269} | age==0 ->NaN: 1



=== Information Value (sorted) ===
                             feature       IV  n_bins  has_missing_bin
RevolvingUtilizationOfUnsecuredLines 1.106690       6                0
             NumberOfTimes90DaysLate 0.800583       2                0
NumberOfTime30-59DaysPastDueNotWorse 0.707032       3                0
                                 age 0.260260       6                0
     NumberOfOpenCreditLinesAndLoans 0.075972       4                0
                       MonthlyIncome 0.074473       5                1
                  NumberOfDependents 0.035303       4                1
                           DebtRatio 0.012620       3                0
        NumberRealEstateLoansOrLines 0.005167       2                0
NumberOfTime60-89DaysPastDueNotWorse 0.000192       1                0

IV threshold = 0.020 | kept 7 / 10 features
Dropped (IV < thr): ['DebtRatio', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse']

WoE design matrix: (150000, 8) 

Saved figure: 01_woe_curves (png+pdf)

01 complete.
